<a href="https://colab.research.google.com/github/RCalvoso/grupo4_projeto_integrador_2/blob/main/07_Pipeline_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
from transformers import AutoTokenizer, AutoModel
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from google.colab import drive

# 1. Montar Drive e Carregar Modelo
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

dir_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train_clean.csv' in files:
        dir_path = root
        break

# Carregar modelo treinado
model = tf.keras.models.load_model(os.path.join(dir_path, 'modelo_multimodal_final.keras'))

# Carregar encoders e pré-processador dos dados de treino
train_df = pd.read_csv(os.path.join(dir_path, 'train_clean.csv'))
ignored_cols = ['ticket_id', 'title', 'description', 'image_path', 'target_category', 'created_at', 'updated_at']
feature_cols = [c for c in train_df.columns if c not in ignored_cols]

X_train_num = train_df[feature_cols]
num_cols = X_train_num.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train_num.select_dtypes(include=['object', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])
preprocessor.fit(X_train_num)

classes = np.unique(train_df['target_category'])

# Modelos extratores pré-treinados
resnet = ResNet50(weights='imagenet', include_top=False, pooling='avg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
bert_model = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased").to(device)
bert_model.eval()

# 2. Pipeline de Inferência Fim-a-Fim
def predict_ticket_category(ticket_data):
    # A. Tabular
    df_single = pd.DataFrame([ticket_data[feature_cols]])
    tab_feat = preprocessor.transform(df_single)

    # B. Texto (BERTimbau)
    full_text = str(ticket_data.get('title', '')) + " " + str(ticket_data.get('description', ''))
    encoded = tokenizer([full_text], padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = bert_model(**encoded)
        mask = encoded['attention_mask'].unsqueeze(-1)
        token_embeds = outputs.last_hidden_state
        sum_embeds = torch.sum(token_embeds * mask, dim=1)
        sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        text_feat = (sum_embeds / sum_mask).cpu().numpy()

    # C. Imagem (ResNet50)
    img_path = ticket_data.get('image_path', '')
    if img_path and os.path.exists(img_path):
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        img_feat = resnet.predict(x, verbose=0).flatten().reshape(1, -1)
    else:
        img_feat = np.zeros((1, 2048))

    # Fusão
    X_input = np.hstack([tab_feat, text_feat, img_feat])

    # Predição
    probs = model.predict(X_input, verbose=0)[0]
    pred_idx = np.argmax(probs)

    return {
        'categoria_prevista': classes[pred_idx],
        'confianca': float(probs[pred_idx]),
        'probabilidades': {classes[i]: float(probs[i]) for i in range(len(classes))}
    }

print("✅ Pipeline de inferência pronto para teste!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Pipeline de inferência pronto para teste!
